[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/affine_toda/affine_toda.ipynb)

# E8 Toda Masses and Interactions

This example computes the particle mass spectrum and three-particle cubic couplings of affine E8 Toda field theory from the geometry of the E8 root system. The Toda field in one spatial and one time dimension is modelled as taking values in the eight-dimensional space of E8's roots, where the curvature of a local exponential potential at equilibrium sets the particle masses and its third derivative selects which particle triples interact. As described by [Koca and Koca](https://arxiv.org/abs/1204.4567) and by [Dorey's root-triangle rule](https://arxiv.org/abs/hep-th/9110058), the particle masses stand in the exact ratios of the Coxeter ring radii, and three species couple exactly when roots from their rings close into a triangle whose projected area sets the coupling strength.

The field takes intrinsic `Antivector` values while each root acts as an extrinsic `Vector` mirror that reads the field by pairing via `roots & Field`. Because the potential sums exponentials of those scalar readings, its quadratic and cubic derivatives at zero field assemble directly as multilinear forms `(marks * reading * reading).sum(axis=0)` and `(marks * reading * reading * reading).sum(axis=0)`, and wedge areas in the Coxeter plane follow from `(mass_triangles[:, 0] ^ mass_triangles[:, 1]).scalar_product(coxeter_plane)`. The root orbits and Coxeter plane come from [E8 Reflections](../../math/e8/e8.ipynb).

In Cartesian component notation on $\mathbb{R}^8$, choosing an orthonormal coordinate basis for the root vectors $\alpha_i = (\alpha_i^1, \dots, \alpha_i^8)$ and field components $\phi = (\phi^1, \dots, \phi^8)$ writes the potential with Kac marks $n_i$ and negative highest root $\alpha_0 = -\sum_{i=1}^8 n_i \alpha_i$ as $V(\phi) = \frac{m^2}{\beta^2}\sum_{i=0}^{8} n_i \left(e^{\beta \sum_{a=1}^8 \alpha_i^a \phi^a} - 1\right)$, and expanding in component partial derivatives around $\phi = 0$ builds the coordinate mass matrix $M^2_{ab} = \left.\frac{\partial^2 V}{\partial \phi^a \partial \phi^b}\right|_0 = m^2 \sum_{i=0}^8 n_i\, \alpha_i^a \alpha_i^b$ and rank-three coupling tensor $C_{abc} = \left.\frac{\partial^3 V}{\partial \phi^a \partial \phi^b \partial \phi^c}\right|_0 = m^2 \beta \sum_{i=0}^8 n_i\, \alpha_i^a \alpha_i^b \alpha_i^c$, which must then be contracted across three eigenvector index slots $C_{ijk} = \sum_{a,b,c} C_{abc}\, v_i^a v_j^b v_k^c$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, concatenate, stack
from examples.math.e8 import core as e8
from examples.math.e8.scenarios import root_system
from examples.quantum.affine_toda import render, scenarios
from examples.util.animation import save_figure

ga = Algebra("a+b+c+d+e+f+g+h+")
Scalar = ga.gatype.scalar()      # Scalar+
Root = ga.gatype.vector()        # Vector+
Bivector = ga.gatype.bivector()  # Bivector+
# The field's values are antivectors; the roots that read them are mirrors, vectors.
Field = ga.gatype.antivector()   # Antivector-
MassForm = ga.gatype((Scalar, Field, Field))
CubicForm = ga.gatype((Scalar, Field, Field, Field))

## Vacuum curvature and mass spectrum

The Toda potential sums exponentials of the field read by nine roots: the eight simple roots together with the negative highest root, weighted by the Kac marks so that the marked sum of roots vanishes and zero field is an equilibrium. Because each exponential's second derivative at zero is the square of its root's linear reading, the vacuum curvature is a symmetric bilinear form on the field whose eigenvalues give the eight squared masses at unit mass and coupling scale.

In [ ]:
marks = scenarios.marks
root_orbits, plane_x, plane_y, planes, *_ = root_system()                   # [30, 8] Root+, [4] Root+, [4] Root+, [4] Bivector+
horizontal = plane_x[0]                                                     # [] Root+
vertical = plane_y[0]                                                       # [] Root+
coxeter_plane = planes[0]                                                   # [] Bivector+

In [ ]:
def interaction_roots(simple_roots: Root, marks: np.ndarray) -> Root:
    """The eight simple roots and the negative highest root (`[9] Root+`) have a vanishing marked sum at the vacuum."""
    affine_root = -(marks[:-1] * simple_roots).sum(axis=0)                  # [] Root+
    return concatenate((simple_roots, affine_root[None]))                   # [9] Root+


def potential(field: Field, roots: Root, marks: np.ndarray) -> Scalar:
    """The marked Toda potential density relative to the vacuum (`[...] Scalar+`) sums each root's exponentiated reading less one."""
    return (marks * ((roots & field[..., None]).exp() - 1)).sum(axis=-1)    # [...] Scalar+


roots = interaction_roots(e8.simple_roots(), marks)                         # [9] Root+
vacuum_potential = potential(0 * roots[0].dual(), roots, marks)             # [] Scalar+

In [ ]:
def vacuum_forms(roots: Root, marks: np.ndarray) -> tuple[MassForm, CubicForm]:
    """The marked products of root readings give the quadratic mass form and cubic interaction form on the field."""
    reading = roots & Field                                                 # [9] Scalar+ <- Field-
    curvature = (marks * reading * reading).sum(axis=0)                     # [] Scalar+ <- (Field-, Field-)
    cubic = (marks * reading * reading * reading).sum(axis=0)               # [] Scalar+ <- (Field-, Field-, Field-)
    return curvature, cubic


def mass_spectrum(curvature: MassForm, root_orbits: Root,
                  coxeter_plane: Bivector) -> tuple[Scalar, Field, Root, Scalar]:
    """The vacuum curvature's eigenvalues give the eight particle masses (`[8] Scalar+`), ordered alongside the Coxeter ring radii."""
    eigenvalues, modes = curvature.eigh()                                   # [8] Scalar+, [8] Field-
    masses = eigenvalues.square_root()                                      # [8] Scalar+
    radii = (root_orbits[0] | coxeter_plane).scalar_norm_squared().square_root()  # [8] Scalar+
    order = radii.argsort()
    return masses, modes, root_orbits[:, order], radii[order]


curvature, cubic = vacuum_forms(roots, marks)                               # [] Scalar+ <- (Field-, Field-), [] Scalar+ <- (Field-, Field-, Field-)
masses, modes, root_orbits, radii = mass_spectrum(curvature, root_orbits, coxeter_plane)  # [8] Scalar+, [8] Field-, [30, 8] Root+, [8] Scalar+
print("masses over the lightest:", (masses / masses[0]).to_array())
print("radii over the smallest:", (radii / radii[0]).to_array())
masses_figure = render.draw_roots(root_orbits, horizontal, vertical)
save_figure(masses_figure, "affine_toda_masses")
masses_figure

The eight particle mass ratios match the ratios of the projected Coxeter ring radii, with colours ordering the species from lightest to heaviest: yellow, red, pink, violet, blue, cyan, green, and lime. Dots mark individual projected roots, while faint chords connect nearest neighbours in the full eight-dimensional space.

## Measured excitation energies

Inelastic neutron scattering measures the energy required to excite spin modes in quasi-one-dimensional magnets, and at the zone centre the E8 Ising scaling theory predicts the same mass ratios as affine E8 Toda theory. [Wang and colleagues, Table I](https://arxiv.org/pdf/2308.00249#page=3), report the lowest three excitation energies measured on IN12 in BaCo₂V₂O₈ at a transverse field of 4.7 T, a temperature of 1.5 K, and wavevector (0, 0, 2); fixing the overall energy scale to the lightest peak predicts the next two from the curvature's mass ratios alone.

In [ ]:
measured_energies = scenarios.measured_energies
n_measured = len(measured_energies)

# Only the lightest peak sets the scale; the remaining peaks follow from the Toda mass ratios.
predicted_energies = measured_energies[0] * masses[:n_measured] / masses[0]  # [n_measured] Scalar+
print("E8 predictions, meV:", predicted_energies.to_array())
print("measured peaks, meV:", measured_energies)

The second and third predicted energies lie within a percent of the Gaussian-fitted experimental peak centres, with the third peak sitting near the two-particle continuum threshold. Whereas this experiment probes the shared E8 mass ratios in the scaling limit, the three-mode couplings below are evaluated directly within classical Toda theory.

## Three-mode couplings and root triangles

Evaluating the cubic form `cubic` on three mass eigenmodes gives their three-particle coupling strength, which is nonzero precisely when one root from each of the three species' Coxeter orbits sums to zero in eight dimensions. Among the triples 1–1–2, 1–2–3, and 2–3–5, the third has zero coupling even though its three scalar masses satisfy the ordinary triangle inequalities.

In [ ]:
species = scenarios.example_species
n_cases = len(species)

In [ ]:
def root_triangles(roots: Root, species: np.ndarray) -> tuple[Root, Scalar]:
    """Each species triple selects a closest-closing root triangle (`[n_triples, 3] Root+`) with squared closure gap (`[n_triples] Scalar+`)."""
    n_steps = roots.shape[0]
    n_triples = len(species)
    step_indices = np.arange(n_steps)
    triple_indices = np.arange(n_triples)

    # A common Coxeter turn preserves closure, so the first root stays at step zero.
    first_roots = roots[0, species[:, 0]]                                   # [n_triples] Root+
    second_roots = roots[step_indices[None, :], species[:, 1, None]]        # [n_triples, n_steps] Root+
    third_roots = roots[step_indices[None, :], species[:, 2, None]]         # [n_triples, n_steps] Root+
    residuals = first_roots[:, None, None] + second_roots[:, :, None] + third_roots[:, None, :]  # [n_triples, n_steps, n_steps] Root+
    closure = residuals.scalar_norm_squared().reshape(n_triples, -1)        # [n_triples, n_steps * n_steps] Scalar+
    nearest = closure.argmin(axis=-1)
    triangles = stack((first_roots, second_roots[triple_indices, nearest // n_steps],
                       third_roots[triple_indices, nearest % n_steps]), axis=1)  # [n_triples, 3] Root+
    return triangles, closure[triple_indices, nearest]


couplings = cubic(modes[species[:, 0]], modes[species[:, 1]], modes[species[:, 2]])  # [n_cases] Scalar+
triangles, closure = root_triangles(root_orbits, species)                   # [n_cases, 3] Root+, [n_cases] Scalar+
print("species:", species + 1)
print("|coupling|:", couplings.abs().to_array())
print("root closure:", closure.square_root().to_array())
triangles_plot = render.triangles_figure(triangles, root_orbits, species, horizontal, vertical)
save_figure(triangles_plot, "affine_toda_triangles")
triangles_plot

Coloured arrows place one root from each species head to tail starting at the white dot. The first two triples return cleanly to the origin, whereas the third leaves a dashed white gap even for its closest-closing combination of orbit roots, matching its vanishing cubic coupling because the discrete root lattice imposes a stricter selection rule than scalar mass lengths alone.

## Coupling strength from projected triangle area

Whenever three roots close into a triangle in eight dimensions, scaling the triangle by the ratio of mass to orbit radius and reading its projected wedge area in the Coxeter plane gives the magnitude of the cubic coupling directly.

In [ ]:
def area_couplings(triangles: Root, mass_scale: Scalar, coxeter_plane: Bivector) -> Scalar:
    """Projected wedge areas of mass-scaled root triangles in the Coxeter plane determine the cubic coupling magnitudes (`[...] Scalar+`)."""
    mass_triangles = triangles * mass_scale                                 # [..., 3] Root+
    areas = (mass_triangles[:, 0] ^ mass_triangles[:, 1]).scalar_product(coxeter_plane).abs() / 2  # [...] Scalar+
    return 4 * areas / np.sqrt(e8.coxeter_order)                            # [...] Scalar+


closed = closure < 1e-6
predicted = area_couplings(triangles[closed], masses[0] / radii[0], coxeter_plane)  # [...] Scalar+
print("from the cubic form:", couplings[closed].abs().to_array())
print("from triangle area:", predicted.to_array())

Both evaluations produce identical coupling magnitudes for every allowed triple, uniting the potential's quadratic curvature, its cubic interaction tensor, and projected root-triangle areas in one Coxeter-plane geometry.

In [ ]:
# checks
assert (vacuum_potential.abs() < 1e-8).all()
assert ((radii / radii[0] - masses / masses[0]).abs() < 1e-8).all()
assert ((predicted_energies - measured_energies).abs() < 0.03).all()
np.testing.assert_array_equal(closed, couplings.abs() > 1e-6)
assert ((couplings[closed].abs() - predicted).abs() < 1e-8).all()